# Lab 03: File API and Long Context Window (Gemini 3.1)

Gemini 3.1 models feature an industry-leading context window (up to 2M tokens). In this lab, we will use the **File API** to process multiple documents and manage our remote storage.

## Objectives
1. Upload multiple files using `client.files.upload`.
2. Perform **Cross-document Reasoning** (analyzing multiple files at once).
3. List and audit files in remote storage.
4. Manage file cleanup.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Uploading Multiple Files

We will create two separate text files and upload them to Gemini's File API storage.

In [ ]:
# Create sample documents
docs = {
    "project_alpha.txt": "Project Alpha is focused on building a Lunar base by 2030.",
    "project_beta.txt": "Project Beta is a mission to explore Mars' subsurface"
    "water by 2035."
}

uploaded_files = []
for filename, content in docs.items():
    with open(filename, "w") as f:
        f.write(content)

    print(f"Uploading {filename}...")
    file_obj = client.files.upload(file=filename, config={"display_name": filename})
    uploaded_files.append(file_obj)

print(f"\nSuccessfully uploaded {len(uploaded_files)} files.")

## 2. Cross-document Reasoning

Gemini can reason across all files provided in the `contents` list. This is much more efficient than traditional RAG for medium-sized datasets.

In [ ]:
prompt="Compare Project Alpha and Project Beta. What are their goals and target dates?"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[
        *uploaded_files, # Unpack the list of file objects
        prompt
    ]
)

display(Markdown(response.text))

## 3. Storage Audit and Management

Files uploaded to the File API are stored for **48 hours**. You can list them to check your current remote storage usage.

In [ ]:
print("Current files in your Gemini remote storage:")
for f in client.files.list():
    print(f"- {f.display_name} (Name: {f.name}, URI: {f.uri})")

## 4. Systematic Cleanup

It is a best practice to delete files once you are done with them, both locally and remotely.

In [ ]:
print("Cleaning up...")
for f in uploaded_files:
    # Delete remote file
    client.files.delete(name=f.name)
    # Delete local file
    if os.path.exists(f.display_name):
        os.remove(f.display_name)
    print(f"Deleted {f.display_name}")

print("\nCleanup complete.")

## Summary

In this lab, you mastered the File API for large context reasoning:
1. Handling multiple uploads.
2. **Reasoning across multiple files** simultaneously.
3. Auditing and **managing remote storage** cycle of life.